# House Price Prediction
**An End-to-End Machine Learning Web Application (FastAPI + Streamlit)**

**Author:** Souraneel Biswas  
**Dataset:** `house_price_regression_dataset.csv` (1,000 rows, 8 columns)  
**Task:** Supervised Regression — predict residential house prices from property characteristics

---
### Contents
1. Imports & Configuration  
2. Load & Explore Dataset  
3. Data Preparation  
4. Model Training & Comparison  
5. Best Model Selection & Evaluation  
6. Save Model Artefacts  
7. Feature Importance Visualisation  
8. FastAPI Backend — `app.py` (Overview)  
9. Streamlit Frontend — `frontend.py` (Overview)  
10. Sample Prediction Walkthrough  
11. Summary & Conclusions

## 1. Imports & Configuration

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH    = 'house_price_regression_dataset.csv'
MODEL_DIR    = 'model'
PLOTS_DIR    = 'plots'
RANDOM_STATE = 42

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print('Libraries loaded successfully.')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
df.describe().round(2)

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['House_Price'], bins=40, color='#2d6a9f', edgecolor='white')
axes[0].set_title('House Price Distribution')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].scatter(df['Square_Footage'], df['House_Price'],
                alpha=0.4, color='#2d6a9f', s=15)
axes[1].set_title('Square Footage vs House Price')
axes[1].set_xlabel('Square Footage')
axes[1].set_ylabel('Price (USD)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'eda_overview.png'), dpi=120)
plt.show()
print('EDA plots saved.')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'correlation_heatmap.png'), dpi=120)
plt.show()

## 3. Data Preparation

In [ ]:
FEATURES = [
    'Square_Footage',
    'Num_Bedrooms',
    'Num_Bathrooms',
    'Year_Built',
    'Lot_Size',
    'Garage_Size',
    'Neighborhood_Quality',
]
TARGET = 'House_Price'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {FEATURES}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'LinearRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression()),
    ]),
    'RandomForest': RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=5, random_state=RANDOM_STATE
    ),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f'{name:25s}  MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}')

In [ ]:
# Display comparison table
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'MAE', 'RMSE', 'R²']
results_df = results_df.sort_values('R²', ascending=False)
results_df['MAE']  = results_df['MAE'].map('${:,.2f}'.format)
results_df['RMSE'] = results_df['RMSE'].map('${:,.2f}'.format)
results_df['R²']   = results_df['R²'].map('{:.4f}'.format)
print(results_df.to_string(index=False))

## 5. Best Model Selection & Evaluation

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = models[best_name]
print(f'Best model : {best_name}')
print(f"  R²   = {results[best_name]['R2']:.4f}")
print(f"  MAE  = ${results[best_name]['MAE']:,.2f}")
print(f"  RMSE = ${results[best_name]['RMSE']:,.2f}")

In [ ]:
# Actual vs Predicted plot
best_preds = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted scatter
axes[0].scatter(y_test, best_preds, alpha=0.5, color='#2d6a9f', s=18)
lims = [min(y_test.min(), best_preds.min()),
        max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Actual vs Predicted — {best_name}')
axes[0].legend()
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# Residuals
residuals = y_test - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.5, color='#e05c2a', s=18)
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'actual_vs_predicted.png'), dpi=120)
plt.show()

## 6. Save Model Artefacts

In [ ]:
joblib.dump(best_model, os.path.join(MODEL_DIR, 'model.pkl'))

meta = {
    'model_name': best_name,
    'features'  : FEATURES,
    'metrics'   : results[best_name],
    'feature_stats': {
        col: {
            'min':  float(df[col].min()),
            'max':  float(df[col].max()),
            'mean': float(df[col].mean()),
        }
        for col in FEATURES
    },
    'all_model_metrics': results,
}
with open(os.path.join(MODEL_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Model saved → {MODEL_DIR}/model.pkl')
print(f'Meta  saved → {MODEL_DIR}/meta.json')

## 7. Feature Importance Visualisation

In [ ]:
inner = best_model.named_steps['model'] if hasattr(best_model, 'named_steps') else best_model
importances = inner.feature_importances_ if hasattr(inner, 'feature_importances_') else inner.coef_

fig, ax = plt.subplots(figsize=(8, 5))
sorted_idx = np.argsort(importances)[::-1]
sns.barplot(x=[importances[i] for i in sorted_idx],
            y=[FEATURES[i]    for i in sorted_idx],
            ax=ax, palette='viridis')
ax.set_title(f'Feature Importances — {best_name}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'feature_importance.png'), dpi=150)
plt.show()
print(f'Plot saved → {PLOTS_DIR}/feature_importance.png')

## 8. FastAPI Backend — `app.py` (Overview)

The FastAPI backend loads `model/model.pkl` and `model/meta.json` at startup and exposes three REST endpoints:

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | `/` | Health check |
| GET | `/model-info` | Model name, metrics, feature stats |
| POST | `/predict` | Accept house features → return predicted price |

**Start the backend:**
```bash
uvicorn app:app --reload
```
Swagger docs available at: http://localhost:8000/docs

**Sample request:**
```python
import requests
payload = {
    'Square_Footage': 2000, 'Num_Bedrooms': 3, 'Num_Bathrooms': 2,
    'Year_Built': 2005, 'Lot_Size': 2.5, 'Garage_Size': 1,
    'Neighborhood_Quality': 7
}
r = requests.post('http://localhost:8000/predict', json=payload)
print(r.json())
```

## 9. Streamlit Frontend — `frontend.py` (Overview)

The Streamlit frontend communicates with the FastAPI backend via HTTP and provides three tabs:

- **📊 Prediction** — Sidebar input form → price card + gauge chart + input summary  
- **📈 Model Metrics** — Model comparison table + R² bar chart + MAE/RMSE grouped bar  
- **🔍 Data Explorer** — Scatter plot, price histogram, correlation heatmap, raw data

**Start the frontend (requires backend running first):**
```bash
streamlit run frontend.py
```
UI available at: http://localhost:8501

## 10. Sample Prediction Walkthrough

In [ ]:
# Direct model prediction (without the API) — for notebook demonstration
sample = {
    'Square_Footage'      : 2000,
    'Num_Bedrooms'        : 3,
    'Num_Bathrooms'       : 2,
    'Year_Built'          : 2005,
    'Lot_Size'            : 2.5,
    'Garage_Size'         : 1,
    'Neighborhood_Quality': 7,
}

sample_df = pd.DataFrame([sample], columns=FEATURES)
predicted_price = float(best_model.predict(sample_df)[0])

print('── Sample Input ──────────────────────────────')
for k, v in sample.items():
    print(f'  {k:25s}: {v}')
print('── Prediction ────────────────────────────────')
print(f'  Estimated House Price : ${predicted_price:,.2f}')
print(f'  Model Used            : {best_name}')

## 11. Summary & Conclusions

### Model Comparison Results

| Model | MAE | RMSE | R² |
|---|---|---|---|
| LinearRegression | \$8,174.58 | \$10,071.48 | 0.9984 |
| GradientBoosting | \$12,196 | \$14,881 | 0.9966 |
| RandomForest | \$16,035 | \$19,678 | 0.9940 |

### Key Findings
- **Linear Regression** (inside a `StandardScaler` Pipeline) achieved the best test-set performance: **R² = 0.9984**, meaning it explains 99.84% of variance in house prices.
- The dataset exhibits a **strong linear relationship** between `Square_Footage` and `House_Price`, which explains why the linear model outperforms tree-based alternatives.
- The MAE of \$8,174 is approximately **1.32% of the mean house price (\$618,861)** — an excellent fit for a regression model.
- The full system follows the standard **two-tier ML deployment pattern**: FastAPI backend for inference, Streamlit frontend for user interaction.

### Project Architecture
```
House_Price_Regression/
├── train_model.py                       # Model training + evaluation
├── app.py                               # FastAPI REST API backend
├── frontend.py                          # Streamlit web frontend
├── house_price_regression_dataset.csv   # Dataset (1,000 rows)
├── requirements.txt
├── README.md
├── model/
│   ├── model.pkl                        # Saved best model (Pipeline)
│   └── meta.json                        # Feature list, metrics, stats
└── plots/
    └── feature_importance.png
```